# **NEUROCHIP SMART FAULT MAPPER**

### Detect and pinpoint faults in a VLSI design using a Graphical Neural Network.

Author: Nikhil Bhaktha

GitHub Profile: [GlobosNik](https://github.com/GlobosNik)

---
# **Data Extractor Notebook**

## Load the required python modules

Run the following commands in the bash terminal.

In [1]:
# Install Pyverilog for parsing
!pip install pyverilog

# Install iverilog for preprocessing Verilog files (dependency for Pyverilog)
!sudo apt-get install -y iverilog

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (11.0-1.1).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.


Import the required python dependencies.

In [2]:
import pandas as pd
import re
from tqdm import tqdm
import os
from collections import defaultdict, Counter
from pathlib import Path
from pyverilog.vparser.parser import parse
import concurrent.futures
import threading
import functools
import ast
import uuid
import time
from concurrent.futures import ProcessPoolExecutor
import traceback
from concurrent.futures import ThreadPoolExecutor


## Clone the HDL Benchmarks Repository

Clone the HDL Benchmarks GitHub repository to access the dataset.

GitHub Link: [HDL_Benchmarks](https://github.com/ispras/hdl-benchmarks)


In [3]:
# Remove directory if it exists from previous runs
!rm -rf HDL_Benchmarks

# Clone the Benchmarks repository
!git clone https://github.com/ispras/hdl-benchmarks {'HDL_Benchmarks'}

Cloning into 'HDL_Benchmarks'...
remote: Enumerating objects: 8419, done.
remote: Counting objects: 100% (1621/1621), done.
remote: Compressing objects: 100% (1313/1313), done.
remote: Total 8419 (delta 488), reused 1028 (delta 279), pack-reused 6798 (from 1)
Receiving objects: 100% (8419/8419), 47.51 MiB | 8.46 MiB/s, done.
Resolving deltas: 100% (3430/3430), done.
Updating files: 100% (5181/5181), done.


## Normalize and Encode each `.v` File

This function extracts the signal names from declarations/gates/assigns etc. and encodes them into a standard format.
* It takes a raw Verilog text, removes comments, identifies all signal names from declarations (like `wire`, `reg`, `input`, `output`), gates (like `and`, `or`), and assignments (`assign`).
* It then creates a mapping to replace these original signal names with a standardized format, `S0`, `S1`, `S2` etc.
* This standardization is crucial for consistent processing and analysis of VLSI designs.
* The regular expressions at the beginning are pre-compiled patterns used by the function to efficiently find and extract these different types of signal declarations and usages.

In [ ]:
# Pre-compile regex patterns for verilogEncoder
_re_text_no_comments = re.compile(r'/\*.*?\*/|//.*?(?=\n|$)', flags=re.DOTALL | re.MULTILINE)
_re_decl_pattern     = re.compile(r'(?:wire|reg|input|output)\s*(?:\[.*?\])?\s*([a-zA-Z_]\w*(?:\s*,\s*[a-zA-Z_]\w*)*?)(?=\s*;)')
_re_gate_pattern     = re.compile(r'(?:and|or|not|xor|nor|xnor|buf|inv)\s*\(\s*([^)]+)\)')
_re_assign_pattern   = re.compile(r'assign\s+([a-zA-Z_]\w*)\s*=\s*([^;]+);')
_re_findall_signals  = re.compile(r'[a-zA-Z_]\w+')
_re_any_ident        = re.compile(r'\b([a-zA-Z_]\w*)\b')

def verilogEncoder(text):
    text_no_comments = _re_text_no_comments.sub('', text)

    signals = set()

    # ONLY collect from structured declarations/gates/assigns — no broad scan
    for pat_compiled in [_re_decl_pattern, _re_gate_pattern, _re_assign_pattern]:
        for match in pat_compiled.finditer(text_no_comments):
            if pat_compiled == _re_assign_pattern:
                signals.add(match.group(1))
                signals.update(_re_findall_signals.findall(match.group(2)))
            else:
                args = match.group(1) if len(match.groups()) > 0 else ''
                signals.update(_re_findall_signals.findall(args))

    signals = {s for s in signals if s and not s.startswith(('S', 's'))}
    sorted_signals = sorted(list(signals), key=len, reverse=True)
    signal_map = {sig: f'S{i}' for i, sig in enumerate(sorted_signals)}

    if not sorted_signals:
        return text_no_comments, signal_map

    # General pattern + dict lookup — O(n)
    normalized = _re_any_ident.sub(
        lambda m: signal_map.get(m.group(0), m.group(0)),
        text_no_comments
    )
    return normalized, signal_map

## Parse the Files and Convert to DataFrames

The `verilogExtractor` function is defined here, which takes normalized Verilog code and converts it into GNN representation suitable for analysis.

- **Regular Expressions:**
    - The pre-compiled regular expressions (`_re_gate_extract_pat`, `_re_assign_extract_pat`, `_re_rhs_signals`, `_re_port_extract_pat`) are defined here.
    - These patterns are used to efficiently identify and extract information about gates (like `and`, `or`), signal assignments (`assign`), signals on the right-hand side of assignments, and input/output port declarations from the normalized Verilog code.
- **Extractor Function:**
    It initializes empty lists for nodes and edges (which will eventually become dataframes), and a dictionary `graph_stats` to store various metrics about the circuit.
- **Signal ID Mapping:**
    - A `signal_to_id` dictionary and `next_id` counter are used to assign a unique numerical ID to each signal (`S0`, `S1`, etc.) found in the Verilog code.
    - The `get_id` helper function ensures that each signal gets a unique ID and consistently maps the same signal name to the same ID.
- **Parsing Gates:**
    - The code iterates through all matches of `_re_gate_extract_pat` in the normalized Verilog.
    - For each gate, it extracts the gate type (`gtype`), output signal (`out_sig`), and input signals (`ins_str`).
    - It then creates a node for the output signal with its type, `fan-in` (number of inputs), and initializes `fan-out` to 0.
    - Edges are created from each input signal to the output signal.
    - The `graph_stats` are updated with the count of gates and gate types.
- **Parsing Assignments:**
    - Similar to gates, this parses `assign` statements using `_re_assign_extract_pat`.
    - It extracts the output signal and the right-hand side (RHS) expression.
    - It then uses `_re_rhs_signals` to find all signals on the RHS, treating them as inputs to the assignment.
- **Parsing Ports:**
    - The `_re_port_extract_pat` is used to identify input and output port declarations.
    - For each port, a node is created with the appropriate type (`input` or `output`) and `is_port flag` set to 1. The `num_inputs` and `num_outputs` in `graph_stats` are updated.
- **Compute Fanout, Degrees, and Depth:** After all nodes and edges are collected, the function calculates the fan-out for each node by counting how many times a node appears as a source in the edges list.

After this, the final dataframes for nodes and edges are returned.

In [5]:
# Pre-compile regex patterns for verilogExtractor
_re_gate_extract_pat = re.compile(r'(and|or|not|xor|nor|xnor|buf|inv)\s*\(\s*([A-Z]\d+)\s*,?\s*([^)]*)\)\s*;')
_re_assign_extract_pat = re.compile(r'assign\s+([A-Z]\d+)\s*=\s*([^;]+);')
_re_rhs_signals = re.compile(r'[A-Z]\d+')
_re_port_extract_pat = re.compile(r'(input|output)\s+([A-Z]\d+(?:\s*,\s*[A-Z]\d+)*);')

# Parse normalized Verilog into GCN features
def verilogExtractor(normalized_code):
    nodes = []  # [node_id, type, fanin, fanout, is_port]
    edges = []  # [src_id, dst_id]
    graph_stats = {
        'num_gates': 0, 'num_inputs': 0, 'num_outputs': 0,
        'gate_types': Counter(), 'avg_degree': 0, 'max_depth': 0
    }

    signal_to_id = {}  # S0 -> 0
    next_id = 0

    def get_id(sig):
        nonlocal next_id
        if sig not in signal_to_id:
            signal_to_id[sig] = next_id
            next_id += 1
        return signal_to_id[sig]

    # Parse gates
    for match in _re_gate_extract_pat.finditer(normalized_code):
        gtype, out_sig, ins_str = match.groups()
        out_id = get_id(out_sig)
        ins = [s.strip() for s in ins_str.split(',') if s.strip()]
        in_ids = [get_id(sig) for sig in ins]

        nodes.append({'node_id': out_id, 'type': gtype, 'fanin': len(ins), 'fanout': 0, 'is_port': 0})
        graph_stats['gate_types'][gtype] += 1
        graph_stats['num_gates'] += 1

        for in_id in in_ids:
            edges.append({'src': in_id, 'dst': out_id})

    # Parse assigns: assign S1 = S2 & S3;
    for match in _re_assign_extract_pat.finditer(normalized_code):
        out_sig, rhs = match.groups()
        out_id = get_id(out_sig)
        # Simple: extract signals in RHS as inputs
        ins = _re_rhs_signals.findall(rhs)
        in_ids = [get_id(sig) for sig in ins]

        nodes.append({'node_id': out_id, 'type': 'assign', 'fanin': len(ins), 'fanout': 0, 'is_port': 0})
        graph_stats['num_gates'] += 1

        for in_id in in_ids:
            edges.append({'src': in_id, 'dst': out_id})

    # Ports (inputs/outputs)
    for match in _re_port_extract_pat.finditer(normalized_code):
        ptype, ports_str = match.groups()
        ports = [s.strip() for s in ports_str.split(',') if s.strip()]
        for port in ports:
            pid = get_id(port)
            if ptype == 'input':
                graph_stats['num_inputs'] += 1
                nodes.append({'node_id': pid, 'type': 'input', 'fanin': 0, 'fanout': 0, 'is_port': 1})
            else:
                graph_stats['num_outputs'] += 1
                nodes.append({'node_id': pid, 'type': 'output', 'fanin': 0, 'fanout': 0, 'is_port': 1})

    # Compute fanout, degrees, depth
    fanout_count = defaultdict(int)
    for e in edges:
        fanout_count[e['src']] += 1

    for node in nodes:
        node['fanout'] = fanout_count[node['node_id']]

    if nodes:
        degrees = [n['fanin'] + n['fanout'] for n in nodes]
        graph_stats['avg_degree'] = sum(degrees) / len(nodes)
        graph_stats['max_depth'] = max(degrees)

    return pd.DataFrame(nodes), pd.DataFrame(edges), graph_stats

## Extract the Verilog Files

Extract the paths of all verilog (`.v`) files in the repository into a list

In [6]:
# Find all .v files in the HDL_Benchmarks directory
verilog_fileList = list(Path('HDL_Benchmarks').rglob('*.v'))
print(f"Found {len(verilog_fileList)} Verilog files to process.")

Found 2040 Verilog files to process.


## Node Feature Engineering

Implement one-hot encoding for the dataframe parameters, using a `oneHotEncoder` function, which is responsible for transforming and normalizing features in the `nodesDF` dataframe, passed as its parameter.

It performs feature engineering by applying one-hot encoding to the `type` column, creating new binary columns for each gate type. It also normalizes numerical features like `fanin` and `fanout` by scaling them between 0 and 1, and then calculates a normalized degree as the average of the normalized fan-in and fan-out.

In [7]:
# One-hot encoding for gate types
gate_types = ['input', 'output', 'assign', 'and', 'or', 'not', 'xor', 'nor', 'xnor', 'buf', 'inv']
type_to_idx = {t: i for i, t in enumerate(gate_types)}

def oneHotEncoder(nodesDF):
    dummies = pd.get_dummies(nodesDF['type'], prefix='type').reindex(
        columns=[f'type_{t}' for t in gate_types], fill_value=0
    ).astype(int)
    nodesDF = pd.concat([nodesDF, dummies], axis=1)

    nodesDF['fanin_norm']  = nodesDF['fanin']  / (nodesDF['fanin'].max()  + 1e-6)
    nodesDF['fanout_norm'] = nodesDF['fanout'] / (nodesDF['fanout'].max() + 1e-6)
    nodesDF['degree_norm'] = (nodesDF['fanin_norm'] + nodesDF['fanout_norm']) / 2
    return nodesDF

## Multithreading Setup for Faster Data Extraction

This function defines the multithreading process for extracting and transforming data from the verilog (`.v`) files.

- `get_optimal_workers(num_files)`: This is a utility function that helps determine the number of worker processes to use for parallel processing. It aims to use a maximum of 16 workers, but will not exceed the number of files if there are fewer than 16.
- `chunkCreator(lst, chunk_size)`: This is a generator function that takes a list and a chunk_size and yields smaller sub-lists (chunks) from the original list. This is useful for dividing a large list of files into manageable batches for processing.
- `verilogProcessor(filepath)`: This is the core function that processes a single Verilog (.v) file in the following steps:
  - Initialization: It generates a unique `circuit_id` (using `uuid.uuid4().hex`) for the current Verilog file and initializes a `result_status` dictionary to store metadata and processing times.
  - File Reading: It reads the content of the Verilog file from the given filepath. It includes error handling for potential UnicodeDecodeError by attempting to read with utf-8 first and then falling back to latin-1 encoding.
  - Verilog Encoding: It calls the verilogEncoder function (defined in a previous cell) to normalize the Verilog code. This step renames signals to a standardized format (e.g., `S0`, `S1`, `S2`).
  - Feature Extraction: It then calls the `verilogExtractor` function (also defined earlier) to parse the normalized code and extract graph features, which include nodes (with their `types`, `fan-in`, `fan-out`, etc.) and edges (connections between signals).
  - Data Tagging and Return: It updates the result_status with various graph statistics (like number of gates, inputs, outputs, etc.) and adds the unique circuit_id to both the generated nodes and edges DataFrames. Finally, it returns the `result_status` dictionary, and the `nodes_df` and `edges_df` DataFrames.
  - Error Handling: If any exception occurs during the processing of a file, it catches the error, records it in the result_status dictionary along with the traceback, prints an error message, and returns an empty DataFrame for nodes and edges.

In [ ]:
# CONFIGURATION
def get_optimal_workers(num_files):
    return min(16, num_files) if num_files > 0 else 1

def chunkCreator(lst, chunk_size):
    """Split list into chunks"""
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]

# CORE PROCESSOR
def verilogProcessor(filepath):
    circuit_id = uuid.uuid4().hex

    result_status = {
        'circuit_id': circuit_id,
        'file': os.path.basename(filepath),
        'path': filepath
    }

    try:
        start_time_read = time.time()
        try:
            with open(filepath, 'r', encoding='utf-8', buffering=1024 * 1024) as f:
                content = f.read()
        except UnicodeDecodeError:
            with open(filepath, 'r', encoding='latin-1', buffering=1024 * 1024) as f:
                content = f.read()
        result_status['time_read'] = time.time() - start_time_read

        start_time_encode = time.time()
        normalized_code, signal_map = verilogEncoder(content)
        result_status['time_encode'] = time.time() - start_time_encode
        result_status['signal_map_len'] = len(signal_map)

        start_time_extract = time.time()
        nodes_df, edges_df, graph_stats = verilogExtractor(normalized_code)
        result_status['time_extract'] = time.time() - start_time_extract

        result_status.update(graph_stats)

        # Tag with circuit_id and return DataFrames for nodes and edges
        if not nodes_df.empty:
            nodes_df['circuit_id'] = circuit_id
        if not edges_df.empty:
            edges_df['circuit_id'] = circuit_id

        return result_status, nodes_df, edges_df

    except Exception as e:
        result_status['error'] = str(e)
        result_status['traceback'] = traceback.format_exc()
        print(f"Error processing {filepath}: {e}\n{traceback.format_exc()}")
        return result_status, pd.DataFrame(), pd.DataFrame()

The `batchExtractor` function is defined here, which is responsible for processing a list of Verilog files in batches using multiprocessing for improved performance.
- Batch Information: It starts by printing messages indicating which batch is being processed and how many files are in that chunk.
- Optimal Worker Calculation: It calls get_optimal_workers (defined in the previous cell) to determine the ideal number of parallel processes to use for the current batch, ensuring efficient resource utilization.
- Process Pool Execution: It uses `ProcessPoolExecutor` to create a pool of worker processes. This allows multiple `verilogProcessor` calls to run concurrently across different CPU cores.
- Concurrent File Processing: It maps the `verilogProcessor` function (which handles a single Verilog file from reading to feature extraction) to each filepath in the `file_list_chunk`.
- Result Aggregation: After all files in the chunk are processed, it iterates through the results. For each file:
    - If no error occurred, it appends the `result_status` (metadata), and extends `all_nodes_data` and `all_edges_data` lists with the `nodes_df` and `edges_df` (converted to lists of dictionaries) returned by `verilogProcessor`.
    - If an error occurred, only the result_status (which contains the error details) is appended.
- DataFrame Conversion: It then converts the collected `status_reports`, `all_nodes_data`, and `all_edges_data` into pandas DataFrames: `status_df`, `nodes_df_raw`, and `edges_df`.
- Feature Engineering: If `nodes_df_raw` is not empty, it applies the `oneHotEncoder` function to perform one-hot encoding on categorical features and normalize numerical features in the nodes data, resulting in the final `nodes_df`.

These DataFrames are then saved as `.csv` files.

In [9]:
def batchExtractor(file_list_chunk, batch):
    print(f"Processing batch {batch} with {len(file_list_chunk)} files.")

    optimal_workers = get_optimal_workers(len(file_list_chunk))
    print(f"Using {optimal_workers} workers for batch {batch}.")

    status_reports = []
    all_nodes_data = []
    all_edges_data = []

    with ProcessPoolExecutor(max_workers=optimal_workers) as executor:
        results = list(tqdm(executor.map(verilogProcessor, file_list_chunk), total=len(file_list_chunk), desc=f"Processing Batch {batch}"))

    for result_status, nodes_data, edges_data in results:
        if 'error' not in result_status:
            status_reports.append(result_status)
            # Convert DataFrames to list of dictionaries before extending
            all_nodes_data.extend(nodes_data.to_dict('records'))
            all_edges_data.extend(edges_data.to_dict('records'))
        else:
            status_reports.append(result_status)

    # Convert lists to DataFrames
    status_df = pd.DataFrame(status_reports)
    nodes_df_raw = pd.DataFrame(all_nodes_data)
    edges_df = pd.DataFrame(all_edges_data)

    # Apply one-hot encoding to nodes_df if not empty
    if not nodes_df_raw.empty:
        nodes_df = oneHotEncoder(nodes_df_raw)
    else:
        nodes_df = nodes_df_raw # Keep empty if no data

    # Save to CSV
    status_df.to_csv(f'statusData_{batch}.csv', index=False)
    nodes_df.to_csv(f'nodesData_{batch}.csv', index=False)
    edges_df.to_csv(f'edgesData_{batch}.csv', index=False)

    print(f"\nBatch {batch} processing complete. Data saved to CSVs.")

Batch 1 Extraction

In [10]:
# Extract first 25% of features in batch 1
iterationCount = int(len(verilog_fileList) * 0.25)

batchExtractor(verilog_fileList[:iterationCount], batch=1)

Processing batch 1 with 510 files.
Using 16 workers for batch 1.


Processing Batch 1: 100%|██████████| 510/510 [00:10<00:00, 47.75it/s]



Batch 1 processing complete. Data saved to CSVs.


Batch 2 Extraction

In [11]:
# Extract next 25% of features in batch 2
batchExtractor(verilog_fileList[iterationCount:iterationCount*2], batch=2)

Processing batch 2 with 510 files.
Using 16 workers for batch 2.


Processing Batch 2: 100%|██████████| 510/510 [00:54<00:00,  9.42it/s]



Batch 2 processing complete. Data saved to CSVs.


Batch 3 Extraction

In [12]:
# Extract next 25% of features in batch 3
batchExtractor(verilog_fileList[iterationCount*2:iterationCount*3], batch=3)

Processing batch 3 with 510 files.
Using 16 workers for batch 3.


Processing Batch 3: 100%|██████████| 510/510 [00:04<00:00, 120.03it/s]



Batch 3 processing complete. Data saved to CSVs.


Batch 4 Extraction

In [13]:
# Extract next 25% of features in batch 4
batchExtractor(verilog_fileList[iterationCount*3:], batch=4)

Processing batch 4 with 510 files.
Using 16 workers for batch 4.


Processing Batch 4: 100%|██████████| 510/510 [00:02<00:00, 231.88it/s]



Batch 4 processing complete. Data saved to CSVs.


### Sanity Check for Batch 1 CSV files

In [14]:
# Check statusData_1.csv
try:
    status_df = pd.read_csv('statusData_1.csv')
    print('statusData_1.csv loaded successfully.')
    print(f'Shape of status_df: {status_df.shape}')
    display(status_df.head())
except FileNotFoundError:
    print('statusData_1.csv not found.')
except Exception as e:
    print(f'Error loading statusData_1.csv: {e}')

statusData_1.csv loaded successfully.
Shape of status_df: (510, 13)


,circuit_id,file,path,time_read,time_encode,time_extract,signal_map_len,num_gates,num_inputs,num_outputs,gate_types,avg_degree,max_depth
0,ec28277ec0794649824dc1048e53e65a,F.v,HDL_Benchmarks/iccad-2017/unit5/F.v,0.001831,3.354061,2.342747,24809,24355,0,0,"Counter({'and': 9369, 'not': 9162, 'or': 2315,...",3.323260,247
1,21c4a2c911eb4995b36fd01abc3966f9,G.v,HDL_Benchmarks/iccad-2017/unit5/G.v,0.006502,2.964742,2.452996,21508,21056,0,0,"Counter({'and': 8202, 'not': 6457, 'or': 2496,...",3.396562,140
2,3402726eff994aefb8b3bf19a93d79a1,F.v,HDL_Benchmarks/iccad-2017/unit8/F.v,0.000344,0.266957,0.343058,2694,2512,0,0,"Counter({'and': 853, 'not': 652, 'buf': 365, '...",3.217357,30
3,c2e4a2dc457d4b159e640f4d43962444,G.v,HDL_Benchmarks/iccad-2017/unit8/G.v,0.004476,0.366285,0.461373,3518,3337,0,0,"Counter({'and': 1128, 'not': 1092, 'nor': 464,...",3.177405,47
4,30dc82441d164747af32775c9c77235e,F.v,HDL_Benchmarks/iccad-2017/unit2/F.v,0.000252,0.141401,0.176120,1284,1117,0,0,"Counter({'and': 334, 'not': 291, 'buf': 289, '...",2.967771,32


In [15]:
# Check nodesData_1.csv
try:
    nodes_df = pd.read_csv('nodesData_1.csv')
    print('\nnodesData_1.csv loaded successfully.')
    print(f'Shape of nodes_df: {nodes_df.shape}')
    display(nodes_df.head())
except FileNotFoundError:
    print('nodesData_1.csv not found.')
except Exception as e:
    print(f'Error loading nodesData_1.csv: {e}')


nodesData_1.csv loaded successfully.
Shape of nodes_df: (253130, 20)


,node_id,type,fanin,fanout,is_port,circuit_id,type_input,type_output,type_assign,type_and,type_or,type_not,type_xor,type_nor,type_xnor,type_buf,type_inv,fanin_norm,fanout_norm,degree_norm
0,0,buf,1,12,0,ec28277ec0794649824dc1048e53e65a,0,0,0,0,0,0,0,0,0,1,0,0.000019,0.004578,0.002299
1,2,buf,1,15,0,ec28277ec0794649824dc1048e53e65a,0,0,0,0,0,0,0,0,0,1,0,0.000019,0.005723,0.002871
2,4,buf,1,97,0,ec28277ec0794649824dc1048e53e65a,0,0,0,0,0,0,0,0,0,1,0,0.000019,0.037009,0.018514
3,6,buf,1,41,0,ec28277ec0794649824dc1048e53e65a,0,0,0,0,0,0,0,0,0,1,0,0.000019,0.015643,0.007831
4,8,buf,1,15,0,ec28277ec0794649824dc1048e53e65a,0,0,0,0,0,0,0,0,0,1,0,0.000019,0.005723,0.002871


In [16]:
# Check edgesData_1.csv
try:
    edges_df = pd.read_csv('edgesData_1.csv')
    print('\nedgesData_1.csv loaded successfully.')
    print(f'Shape of edges_df: {edges_df.shape}')
    display(edges_df.head())
except FileNotFoundError:
    print('edgesData_1.csv not found.')
except Exception as e:
    print(f'Error loading edgesData_1.csv: {e}')


edgesData_1.csv loaded successfully.
Shape of edges_df: (753951, 3)


,src,dst,circuit_id
0,1,0,ec28277ec0794649824dc1048e53e65a
1,3,2,ec28277ec0794649824dc1048e53e65a
2,5,4,ec28277ec0794649824dc1048e53e65a
3,7,6,ec28277ec0794649824dc1048e53e65a
4,9,8,ec28277ec0794649824dc1048e53e65a
